# Calibration levels compared by leave-one-out

Compares four ways of estimating the b-value maps, using the noisy scans (`config.SCAN_TYPE`):

- **No correction** - the nominal b-value
- **Scanner level** - calibration maps fitted to all scans
- **Duration level** - calibration maps fitted to the scans with the same gradient duration
- **Sequence level** - the average b-map of the other repeats of the same sequence

For every scan, repeat *k* of every sequence is held out in fold *k* and the prediction uses only the other repeats
(`cal.loo_bmap_predictions`), so training and test data are independent. The correction factor is the measured b-value
map divided by the predicted one (1 = perfect).

Two figures from the same leave-one-out predictions:
1. the distribution of the correction factor across the whole dataset (all scans, all b-values): boxplot and metrics
2. the distribution of the correction factor maps of the b=800 scans for each direction and gradient duration

In [ ]:
import config   # settings shared by all analysis scripts (also makes the root modules importable)

# General libraries
import json
import numpy as np
import pandas as pd
import nibabel as nib

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Custom functions
import registration as reg
import calib_utils as cal

config.PLOT_DIR.mkdir(exist_ok=True)

### LOAD DATA

In [ ]:
# Sequence parameters and the mask
df = pd.read_hdf(config.PARAM_FNAME, key="df", mode="r")
df["row"] = np.arange(df.shape[0]) # position in the parameter file, scan files and cached maps

# Per-scan maps from estimate_calibration.py. Volume i is row i of the parameter file.
bmap_nii = nib.load(config.BMAP_FNAME)
gmap2_nii = nib.load(config.GMAP2_FNAME)
scan_spacing = reg.spacing_from_affine(bmap_nii.affine)

for nii in (bmap_nii, gmap2_nii):
    meta = json.loads(next(e for e in nii.header.extensions if e.get_code() == 40).content.decode("utf-8"))
    if meta["scan_type"] != config.SCAN_TYPE or meta["spline_param"] != config.SPLINE_PARAM:
        raise ValueError(f"Cached maps were made with scan type {meta['scan_type']} and spline_param {meta['spline_param']}; "
                         f"config has {config.SCAN_TYPE} and {config.SPLINE_PARAM} - re-run estimate_calibration.py")

bmap_4d = bmap_nii.get_fdata()
gmap2_4d = gmap2_nii.get_fdata()

preprocess_mask = nib.load(config.MASK_FNAME).get_fdata().astype(bool)

# Create a mask slightly inset from threshold based mask to avoid boundary artefacts.
core_mask = cal.shrink_mask_mm(preprocess_mask, voxel_spacing=scan_spacing, margin_mm=config.SHRINK_DIST_MM)
calib_mask = core_mask

# Diffusion-weighted scans (those with maps in the cache), in order of scan time
is_dwi = np.array([lab["is_dwi"] for lab in meta["volume_labels"]])
dwi_df = df[is_dwi].copy()
dwi_df.sort_values("scan_time", inplace=True, ignore_index=True)

dwi_df["bmap"] = dwi_df["row"].apply(lambda i: bmap_4d[..., i])
dwi_df["gmap2_bspline"] = dwi_df["row"].apply(lambda i: gmap2_4d[..., i].astype(np.float32))
dwi_df["cmap_meas"] = dwi_df["bmap"] / dwi_df["b_value"]

### LEAVE-ONE-OUT AT EACH CALIBRATION LEVEL

In [ ]:
# For each level, leave-one-out predictions of every scan. We keep
#   - the error (measured / predicted b-value) in the mask, for every scan, for the boxplot
#   - the correction factor maps of the b=800 scans, for the distribution figure
levels = {"scanner": "cmap_resid_full", "duration": "cmap_resid_dur", "sequence": "cmap_resid_seqspec"}

# No correction: measured b-value / nominal b-value
error_vecs = {"none": np.concatenate([c[core_mask].astype(np.float32) for c in dwi_df["cmap_meas"].values])}

b800_df = dwi_df[dwi_df["b_value"] >= config.B_EVAL].copy()
for col in levels.values():
    b800_df[col] = None

for level, resid_col in levels.items():
    print(f"{level} level")
    pred_df = cal.loo_bmap_predictions(dwi_df, core_mask, levels=[level], verbose=False)

    errors = []
    n_dropped = 0
    for idx in pred_df.index:
        pred_bmap = pred_df.at[idx, f"pred_bmap_{level}"]
        bmap = dwi_df.at[idx, "bmap"]

        # A prediction of exactly zero (e.g. every other repeat is invalid at that voxel) gives an infinite error: drop these
        with np.errstate(divide="ignore", invalid="ignore"):
            error = bmap[core_mask] / pred_bmap[core_mask]
        n_dropped += np.count_nonzero(~np.isfinite(error))
        errors.append(error[np.isfinite(error)].astype(np.float32))

        if idx in b800_df.index:
            # as figures.ipynb: scanner- and duration-level maps are valid in the calibration mask,
            # the sequence-level map in a mask of the averaged b-map
            mask = reg.ants_mask(pred_bmap, fmt=bool) if level == "sequence" else calib_mask
            cmap_resid = np.zeros_like(mask, dtype=np.float32)
            cmap_resid[mask] = bmap[mask] / pred_bmap[mask]
            b800_df.at[idx, resid_col] = cmap_resid

    error_vecs[level] = np.concatenate(errors)
    print(f"  {n_dropped} non-finite errors dropped")
    del pred_df

b800_df = b800_df.dropna(subset=list(levels.values())) # scans in complete repeats only

### DISTRIBUTION ACROSS THE WHOLE DATASET

In [ ]:
# Long-form dataframe
error_df = pd.DataFrame({
    "Error": np.concatenate([
        error_vecs["none"],
        error_vecs["scanner"],
        error_vecs["duration"],
        error_vecs["sequence"]
    ]),
    "Group": pd.Categorical(
        ["No correction"] * len(error_vecs["none"]) +
        ["Scanner specific"] * len(error_vecs["scanner"]) +
        ["TE specific"] * len(error_vecs["duration"]) +
        ["Sequence specific"] * len(error_vecs["sequence"]))
})
n_test = 4

order = [
    "No correction", 
    "Scanner specific", 
    "TE specific", 
    "Sequence specific"]
wrapped_order = [
    "No\ncorrection",
    "Scanner\nlevel\ncorrection",
    "Duration\nlevel\ncorrection",
    "Sequence\nlevel\ncorrection"]

rocket = sns.color_palette("rocket", n_test+1)
palette = [rocket[i+1] for i in range(n_test)]

fig, ax = plt.subplots(figsize=(5.5, 4.5))
fs_labels = 10
plt.grid(visible=True, axis="y", c="grey", linewidth=1, alpha=0.25)

ax.axhline(0, color="0.35", linestyle="--", c="grey", linewidth=1)
sns.boxplot(
    data=error_df,
    x="Group",
    y="Error",
    hue="Group",
    order=order,
    width=0.5,
    showfliers=False,      # important for very large datasets
    whis=(5, 95),          # often more stable/readable than default 1.5*IQR here
    palette=palette,
    linewidth=1.2,
    boxprops=dict(alpha=1),
    medianprops=dict(color="black", linewidth=1.2),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
    ax=ax
)

ax.set_xlabel("")
ax.set_ylim(0.6, 1.3)
ax.set_xticklabels(wrapped_order, fontsize=fs_labels, style='italic')
ax.set_ylabel(r"c(r)", fontsize=fs_labels)

plt.tight_layout()

fmt = "png"
plot_fname = config.PLOT_DIR / f"calibration_framework_comparison_boxplot_correction_factor.{fmt}"
fig.savefig(fname=plot_fname, dpi=600, format=fmt, bbox_inches="tight", pad_inches=0.01)

In [ ]:
group_metrics = error_df.groupby("Group", observed=True)["Error"].agg(mean="mean", 
                                                  std="std", 
                                                  median="median", 
                                                  q1=lambda x: x.quantile(0.25), 
                                                  q2=lambda x: x.quantile(0.75),
                                                  iqr=lambda x: x.quantile(0.75) - x.quantile(0.25),
                                                  iwr=lambda x: x.quantile(0.95) - x.quantile(0.05))
metrics_fname = config.PLOT_DIR / "calibration_framework_comparison_metrics.csv"
group_metrics.to_csv(metrics_fname, sep=",", header=True, index=True)

### B=800 CORRECTION FACTOR MAPS

In [ ]:
def plot_correction_factor_kdes(plot_df, calib_mask, fname, title=None):
    """
    Distribution of the correction factor (measured b / predicted b) in the calibration mask, for each
    diffusion direction (rows) and calibration level (columns), coloured by gradient duration.
    Copied from figures.ipynb.
    """

    plot_df = plot_df.sort_values(by="scan_time", ignore_index=True)
    dwi_dur = np.sort(plot_df["dwi_grad_duration"].unique())

    n_row = 3
    n_col = 4

    fig, axes = plt.subplots(
        n_row,
        n_col,
        figsize=(19, 9),
        constrained_layout=True,
        gridspec_kw={"width_ratios": [1,1,1,1]},
    )

    # Define font sizes in figure
    fs_plottitle = 16
    fs_axistitle = 10
    fs_legend = 12

    # Define colour maps
    rocket = sns.color_palette("rocket", 13)
    palette = [rocket[i+1] for i in range(12)]

    v_lims = (0.6, 1.4)
    cmap_list = ["cmap_meas", "cmap_resid_full", "cmap_resid_dur", "cmap_resid_seqspec"]
    column_labels = ["No calibration", "Scanner-level calibration", "Duration-level calibration", "Sequence-level calibration"]

    for i, dw_ax in enumerate("XYZ"):

        for j, dur in enumerate(dwi_dur):

            sub_df = plot_df[(plot_df["dwi_direction"]==dw_ax) & (plot_df["dwi_grad_duration"]==dur)].copy()

            for k, cmap_label in enumerate(cmap_list):

                cmap = np.stack(sub_df[cmap_label].values, axis=-1)

                for l, c in enumerate(cmap.transpose(3,0,1,2)):

                    kde_vals = c[calib_mask].flatten()
                    if l == 0:
                        sns.kdeplot(kde_vals, ax=axes[i, k], color=palette[j+3*k], lw=1, clip=v_lims, alpha=0.5+0.5*j/2, fill=False, label=rf"$\delta = {dur*1e3:.0f}\ ms$")
                    else:
                        sns.kdeplot(kde_vals, ax=axes[i, k], color=palette[j+3*k], lw=1, clip=v_lims, alpha=0.5+0.5*j/2, fill=False, legend=False)
        
                axes[i, k].set_xlim(v_lims)
                axes[i, k].set_ylabel("Density")
                axes[i, k].grid(alpha=0.25)
                axes[i, k].spines["top"].set_visible(False)
                axes[i, k].spines["right"].set_visible(False)
                axes[i, k].set_ylabel(None)
                axes[i, k].set_ylim(0, 10)
                
            
                if i==0:
                    axes[i, k].set_title(column_labels[k], fontsize=fs_plottitle, fontstyle="italic", pad=12)
                    axes[i, k].legend(fontsize=fs_legend, frameon=False)
                elif i==2:
                    axes[i, k].set_xlabel(r"$Correction\ factor$", fontsize=fs_axistitle)

        axes[i, 0].text(
            -0.1, 0.5,
            rf"${dw_ax}$",
            transform=axes[i, 0].transAxes,
            fontsize=fs_plottitle,
            fontweight="normal",
            ha="right",
            va="center",
        )

    if title is not None:
        fig.suptitle(title, fontsize=fs_plottitle + 4)

    fig.savefig(fname=fname, dpi=600, format="png")
    plt.show()

In [ ]:
plot_fname = config.PLOT_DIR / "calibration_b800_corr_map_framework_comparison.png"
plot_correction_factor_kdes(b800_df, calib_mask, plot_fname)